# Compare tissue thickness -- merfish vs. lineage

Side-by-side comparison of `08_measure_tissue_thickness.ipynb`'s own results
for a lineage-tracing sample's two sibling acquisitions (`merfish/` +
`lineage/` -- see `resolve_sample_identity`'s own "split layout" note) --
reuses that notebook's cached stitched heatmap and per-FOV z-stacks
directly, no raw image is re-read here. **Requires
`08_measure_tissue_thickness.ipynb` to have already been run for BOTH
acquisitions** (not necessarily at the same time, or with the same
`ROUND_ID`/microscope).

Outputs (two-column figures, each rotated 90 deg CCW, merfish on the left):
- Section 4: tissue thickness heatmap, one shared colorbar.
- Section 5: mosaic at one shared z depth, one shared scale bar.
- Section 6: z-sweep movie, matched by physical depth (not frame index) so
  a z-range/step mismatch between the two acquisitions can't desync the two
  panels -- see that section's own markdown cell for why.

## 1 — Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

%matplotlib inline
# %matplotlib widget  # uncomment for interactive pan/zoom (ipympl) -- roughly
#                       doubles output size vs. inline: the widget's static-
#                       html fallback embeds a second copy of each rendered
#                       image alongside the PNG output

# The two acquisitions compared below are this clone's SAMPLE_DIR and its
# sibling (next cell).
MERCI_DIR = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/after_imaging/)
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config import ExperimentConfig
from MERci.common.metadata import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.acquisition.configs import find_frame_table_for_hal_config, get_fov_geometry
from MERci.analysis.elevation import identify_boundary_fovs, create_z_mosaic, create_paired_movie
from MERci.visualization import get_merci_figures_dir

print(f"MERci import path: {MERCI_DIR / 'src'}")

## 2 — Parameters

In [ ]:
# The two sibling acquisitions of the same lineage-tracing sample -- see
# resolve_sample_identity's own "split layout" note (each has its own
# MERci/ clone + analysis/ tree; their shared parent is the true sample id).
SAMPLE_ROOT = MERCI_DIR.parent.parent   # <sample>/, holding merfish/ and lineage/
MERFISH_SAMPLE_DIR = SAMPLE_ROOT / "merfish"
LINEAGE_SAMPLE_DIR = SAMPLE_ROOT / "lineage"
assert MERFISH_SAMPLE_DIR.parent == LINEAGE_SAMPLE_DIR.parent, "expected sibling acquisitions of the same sample"

# The notebook whose cache is being reused here -- 08_measure_tissue_thickness.ipynb
# saves under this name regardless of its own filename's leading number (see
# that notebook's own Section 2).
SOURCE_NOTEBOOK_NAME = "measure_tissue_thickness"

# Which microscope each acquisition was imaged on (+ optional objective
# override -- None uses that microscope's own default objective). Must match
# what 08_measure_tissue_thickness.ipynb used for that acquisition -- the
# pixel size/crop derived from it has to match the cached data being reused.
MICROSCOPE_MERFISH, OBJECTIVE_MERFISH = "ST2", None
MICROSCOPE_LINEAGE, OBJECTIVE_LINEAGE = "ST2", None

# Which round to use in each acquisition -- by imaging_type (default
# "cells"), or set the matching ROUND_ID_* directly to override. Must match
# what 08_measure_tissue_thickness.ipynb resolved for that acquisition.
ROUND_IMAGING_TYPE = "cells"
ROUND_ID_MERFISH = None
ROUND_ID_LINEAGE = None

# Channel compared -- lineage's own "cells" round also has a 560 nm channel
# (merfish's does not); fixing this at 405 nm compares like with like and
# keeps the Section 6 sweep to the one channel merfish actually has. Must
# match the CHANNEL_NM 08_measure_tissue_thickness.ipynb used for each
# acquisition.
CHANNEL_NM = 405.0

# Must match the DOWNSAMPLE_FACTOR 08_measure_tissue_thickness.ipynb used
# for each acquisition -- the cached per-FOV z-stacks/heatmap are already at
# this resolution, and there is no cached record of what factor made them.
DOWNSAMPLE_FACTOR = 16

# Section 4 (heatmap): shared colorbar range (um) -- display-only, since the
# cached heatmap stores actual thickness values, not values already clipped
# to this bound. Matches merfish's own 08 run's MAX_Z_COLORMAP.
MAX_Z_COLORMAP = 50.0

# Section 5 (single-z mosaic): shared depth + scale bar for both panels.
Z_MOSAIC_UM = 25.0
GIF_SCALEBAR_UM = 1000.0
GIF_PERCENTILE_CLIP = (1.0, 99.0)

# Section 6 (z-sweep movie): every Nth of merfish's own z-planes (Section 6
# below turns this into the shared/lineage-matched depth list), playback
# speed, silent MP4.
GIF_Z_STRIDE = 1
GIF_FRAME_DURATION_MS = 300
MOVIE_FPS = None

# Explicit plot font sizes (NOTEBOOK_GUIDELINES.md #5).
PLOT_TITLE_FONTSIZE    = 14
PLOT_LABEL_FONTSIZE    = 12
PLOT_TICK_FONTSIZE     = 11
PLOT_LEGEND_FONTSIZE   = 10
PLOT_SUPTITLE_FONTSIZE = 15

NOTEBOOK_NAME = "compare_tissue_thickness_merfish_lineage"
# Cache/figures for THIS notebook's own combined outputs live under
# merfish's own analysis tree (the acquisition shown on the left throughout)
# -- same cross-experiment convention as
# 06_map_cells_across_microscopes.ipynb's own SOURCE_DIR-anchored
# cache_dir/figures_dir.
cache_dir = MERFISH_SAMPLE_DIR / "analysis" / "cache" / NOTEBOOK_NAME
cache_dir.mkdir(parents=True, exist_ok=True)
figures_dir = get_merci_figures_dir(MERFISH_SAMPLE_DIR, "after_imaging", NOTEBOOK_NAME)

print(f"merfish : {MERFISH_SAMPLE_DIR}")
print(f"lineage : {LINEAGE_SAMPLE_DIR}")
print(f"Cache   : {cache_dir}")
print(f"Figures : {figures_dir}")

## 3 — Resolve each acquisition's context

Config/metadata/round/grid resolution -- the same cheap, in-memory work
`08_measure_tissue_thickness.ipynb`'s own Sections 1-4 do (no raw image
read) -- plus loading that notebook's already-cached heavy results
(Section 7's per-FOV z-stacks, Section 8's stitched heatmap). Raises a clear
error if `08_measure_tissue_thickness.ipynb` hasn't been (fully) run yet for
that acquisition/round.

In [ ]:
def load_experiment_context(sample_dir, microscope, objective, round_id_override, label):
    pixel_size_um, image_size_px = get_fov_geometry(microscope, objective)
    sample_name, imaging_dir = resolve_sample_identity(sample_dir / "MERci")
    positions_tag = positions_file_tag(sample_name, imaging_dir)

    config = ExperimentConfig.from_sample_dir(
        sample_dir,
        positions_txt  = sample_dir / "positions" / f"positions_{positions_tag}.txt",
        image_suffix   = ".zarr",
        microscope     = microscope,
        pixel_size_um  = pixel_size_um,
        image_size_px  = image_size_px,
    )
    meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                    image_suffix=config.image_suffix)

    target_round_id = round_id_override if round_id_override is not None else meta.round_for_imaging_type(ROUND_IMAGING_TYPE)
    round_info = meta.rounds[target_round_id]

    for s in meta.series_for_round(target_round_id):
        if not s.hal_config:
            continue
        hal_path = Path(config.settings_dir) / s.hal_config
        ft_path = find_frame_table_for_hal_config(hal_path, config.metadata_dir)
        if ft_path and ft_path.exists():
            frame_table = pd.read_csv(ft_path, index_col=0)
            break
    else:
        raise FileNotFoundError(f"No frame table found for {label} round {target_round_id}")

    channel_frames = frame_table[frame_table["color"].round(0) == round(CHANNEL_NM)].sort_values("z")
    if channel_frames.empty:
        raise ValueError(f"No {CHANNEL_NM} nm frames in {label} round {target_round_id}'s frame table.")
    z_um_list = channel_frames["z"].tolist()

    positions = {fov_id: meta.fovs[fov_id].position
                 for fov_id in round_info.fov_files if round_info.fov_files[fov_id]}
    _, _, grid_indices = identify_boundary_fovs(
        positions, config.step_size_um,
        connectivity=config.ffc_connectivity, tolerance_fraction=config.ffc_neighbor_tolerance,
    )

    source_cache_dir = config.analysis_dir / "cache" / SOURCE_NOTEBOOK_NAME
    heatmap_path  = source_cache_dir / f"elevation_heatmap_round{target_round_id}.npy"
    results_csv   = config.analysis_dir / f"tissue_thickness_round{target_round_id}.csv"
    elevation_dir = source_cache_dir / "elevation" / f"round{target_round_id}"
    if not (heatmap_path.exists() and results_csv.exists() and elevation_dir.exists()):
        raise FileNotFoundError(
            f"{label}: 08_measure_tissue_thickness.ipynb has not been (fully) run for round "
            f"{target_round_id} yet -- expected {heatmap_path}, {results_csv}, and {elevation_dir}."
        )

    heatmap = np.load(heatmap_path)
    n_fovs = len(pd.read_csv(results_csv))
    nonzero = heatmap[~np.isnan(heatmap) & (heatmap != 0)]
    full_thickness_um = float(nonzero.max())
    pct_full_thickness = 100.0 * np.mean(nonzero == full_thickness_um)

    stack_paths = {f: elevation_dir / f"fov{f:04d}_stack.npy" for f in sorted(positions)
                   if (elevation_dir / f"fov{f:04d}_stack.npy").exists()}

    return dict(
        label=label, config=config, meta=meta, round_id=target_round_id,
        grid_indices=grid_indices, heatmap=heatmap, n_fovs=n_fovs,
        pct_full_thickness=pct_full_thickness, full_thickness_um=full_thickness_um,
        stack_paths=stack_paths, z_um_list=z_um_list,
    )

In [ ]:
merfish_ctx = load_experiment_context(MERFISH_SAMPLE_DIR, MICROSCOPE_MERFISH, OBJECTIVE_MERFISH, ROUND_ID_MERFISH, "merfish")
lineage_ctx = load_experiment_context(LINEAGE_SAMPLE_DIR, MICROSCOPE_LINEAGE, OBJECTIVE_LINEAGE, ROUND_ID_LINEAGE, "lineage")

for ctx in (merfish_ctx, lineage_ctx):
    print(f"{ctx['label']:8s}: round {ctx['round_id']}, {ctx['n_fovs']} FOV(s), "
          f"{len(ctx['stack_paths'])} cached z-stack(s), "
          f"z range [{min(ctx['z_um_list']):.1f}, {max(ctx['z_um_list']):.1f}] um "
          f"({len(ctx['z_um_list'])} plane(s))")

## 4 — Tissue thickness heatmap, side by side

`08_measure_tissue_thickness.ipynb` Section 8's own heatmap (only -- not its
histogram sibling), one shared colorbar (`MAX_Z_COLORMAP`, same colormap)
across both panels, each rotated 90 deg CCW.

In [ ]:
heatmap_left  = np.rot90(merfish_ctx["heatmap"], k=1)
heatmap_right = np.rot90(lineage_ctx["heatmap"], k=1)

cmap = plt.cm.turbo.copy()
cmap.set_bad("0.85")

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 9))
ax0.imshow(np.ma.masked_invalid(heatmap_left),  cmap=cmap, vmin=0, vmax=MAX_Z_COLORMAP)
im1 = ax1.imshow(np.ma.masked_invalid(heatmap_right), cmap=cmap, vmin=0, vmax=MAX_Z_COLORMAP)

ax0.set_title(f"merfish, {merfish_ctx['n_fovs']} FOVs\n"
              f"{merfish_ctx['pct_full_thickness']:.1f}% full thickness", fontsize=PLOT_TITLE_FONTSIZE)
ax1.set_title(f"lineage, {lineage_ctx['n_fovs']} FOVs\n"
              f"{lineage_ctx['pct_full_thickness']:.1f}% full thickness", fontsize=PLOT_TITLE_FONTSIZE)
for ax in (ax0, ax1):
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)

cbar = fig.colorbar(im1, ax=[ax0, ax1], label="thickness (um)", fraction=0.046, pad=0.04)
cbar.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)

fig.suptitle("Tissue thickness heatmap -- merfish vs. lineage", fontsize=PLOT_SUPTITLE_FONTSIZE)
fig.savefig(figures_dir / f"{NOTEBOOK_NAME}.heatmap_compare.png", dpi=150)
plt.show()

## 5 — Mosaic at a shared z, side by side

`08_measure_tissue_thickness.ipynb` Section 9's own single-z mosaic,
regenerated for both acquisitions from their own cached z-stacks at the SAME
`Z_MOSAIC_UM` with the SAME `GIF_SCALEBAR_UM` (rather than each
acquisition's own, possibly different, notebook-08 parameter choice) --
each panel keeps its own baked-in `"z = ... um"` label + scale bar
(`create_z_mosaic`'s own convention), which now agree with each other
instead of showing two independently-chosen values.

In [ ]:
merfish_mosaic_path = cache_dir / f"z_mosaic_merfish_round{merfish_ctx['round_id']}.png"
lineage_mosaic_path = cache_dir / f"z_mosaic_lineage_round{lineage_ctx['round_id']}.png"

create_z_mosaic(
    merfish_ctx["stack_paths"], merfish_ctx["z_um_list"], merfish_ctx["grid_indices"], merfish_ctx["config"],
    merfish_mosaic_path, z_um=Z_MOSAIC_UM, downsample_factor=DOWNSAMPLE_FACTOR,
    scalebar_um=GIF_SCALEBAR_UM, percentile_clip=GIF_PERCENTILE_CLIP,
)
create_z_mosaic(
    lineage_ctx["stack_paths"], lineage_ctx["z_um_list"], lineage_ctx["grid_indices"], lineage_ctx["config"],
    lineage_mosaic_path, z_um=Z_MOSAIC_UM, downsample_factor=DOWNSAMPLE_FACTOR,
    scalebar_um=GIF_SCALEBAR_UM, percentile_clip=GIF_PERCENTILE_CLIP,
)
print(f"Saved: {merfish_mosaic_path}")
print(f"Saved: {lineage_mosaic_path}")

In [ ]:
img_left  = Image.open(merfish_mosaic_path).convert("L").rotate(90, expand=True)
img_right = Image.open(lineage_mosaic_path).convert("L").rotate(90, expand=True)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(9, 9))
ax0.imshow(np.array(img_left),  cmap="gray"); ax0.set_title("merfish", fontsize=PLOT_TITLE_FONTSIZE)
ax1.imshow(np.array(img_right), cmap="gray"); ax1.set_title("lineage", fontsize=PLOT_TITLE_FONTSIZE)
for ax in (ax0, ax1):
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle(f"Mosaic at z ~ {Z_MOSAIC_UM:.1f} um, scale bar {GIF_SCALEBAR_UM:.0f} um -- merfish vs. lineage",
             fontsize=PLOT_SUPTITLE_FONTSIZE)
fig.savefig(figures_dir / f"{NOTEBOOK_NAME}.z_mosaic_compare.png", dpi=150)
plt.show()

## 6 — z-sweep movie, side by side (depth-matched, single channel)

`08_measure_tissue_thickness.ipynb` Section 11's own z-sweep movie, as one
combined MP4: both panels sweep through the SAME list of physical depths
(built below from the overlap of merfish's and lineage's own imaged z
ranges), and at every target depth each side independently renders its own
*nearest available* frame (`create_paired_movie`) -- so a z-step-size/offset
mismatch between the two acquisitions never lets the two panels drift out
of sync, only approximate by however coarse that side's own z-step is.
`CHANNEL_NM` (405 nm, Section 2) already restricted both sides' own
`z_um_list` to that one channel -- lineage's own "cells" round also has a
560 nm channel, deliberately excluded here so the sweep compares the same
signal on both sides. One shared scale bar (same `GIF_SCALEBAR_UM`/
`DOWNSAMPLE_FACTOR` for both); each side keeps its own fixed display-
intensity scale (its own signal is not directly comparable to the other
acquisition's raw intensity).

In [ ]:
z_merfish = np.asarray(merfish_ctx["z_um_list"])
z_lineage = np.asarray(lineage_ctx["z_um_list"])
lo = max(z_merfish.min(), z_lineage.min())
hi = min(z_merfish.max(), z_lineage.max())

target_z_um_values = [z for z in merfish_ctx["z_um_list"][::GIF_Z_STRIDE] if lo <= z <= hi]

print(f"merfish z range: [{z_merfish.min():.1f}, {z_merfish.max():.1f}] um ({len(z_merfish)} plane(s))")
print(f"lineage z range: [{z_lineage.min():.1f}, {z_lineage.max():.1f}] um ({len(z_lineage)} plane(s))")
print(f"Shared overlap : [{lo:.1f}, {hi:.1f}] um -> {len(target_z_um_values)} target depth(s) (stride {GIF_Z_STRIDE})")
if lo > z_merfish.min() or hi < z_merfish.max():
    print("NOTE: merfish's own z range extends beyond lineage's -- the non-overlapping depths are dropped from the sweep below.")

In [ ]:
movie_frame_cache_merfish = cache_dir / "movie_frames_merfish" / f"round{merfish_ctx['round_id']}"
movie_frame_cache_lineage = cache_dir / "movie_frames_lineage" / f"round{lineage_ctx['round_id']}"
movie_path = figures_dir / f"{NOTEBOOK_NAME}.tissue_elevation_movie_compare.mp4"

create_paired_movie(
    merfish_ctx["stack_paths"], merfish_ctx["z_um_list"], merfish_ctx["grid_indices"], merfish_ctx["config"],
    lineage_ctx["stack_paths"], lineage_ctx["z_um_list"], lineage_ctx["grid_indices"], lineage_ctx["config"],
    target_z_um_values, movie_path,
    downsample_factor=DOWNSAMPLE_FACTOR, fps=MOVIE_FPS, frame_duration_ms=GIF_FRAME_DURATION_MS,
    scalebar_um=GIF_SCALEBAR_UM, percentile_clip=GIF_PERCENTILE_CLIP,
    frame_cache_dir_a=movie_frame_cache_merfish, frame_cache_dir_b=movie_frame_cache_lineage,
)
print(f"Saved: {movie_path}")